# 🐱🐶 Katze oder Hund? -- Bildklassifikation mit Deep Learning

In diesem Notebook trainieren wir ein neuronales Netz, das **Katzen** von **Hunden** unterscheiden kann.

## Lernziele
1. **Daten laden & verstehen** -- Wie werden Bilder fuer ein Netzwerk vorbereitet?
2. **Transfer Learning** -- Ein vortrainiertes Modell anpassen
3. **Training & Evaluation** -- Wie lernt das Modell?
4. **Black Box vs. Glass Box** -- Koennen wir verstehen, *was* das Modell gelernt hat?
5. **Grad-CAM** -- Wo schaut das Modell hin, wenn es entscheidet?

---
## 0. Setup & Imports

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Geraet auswaehlen (GPU > Apple Silicon > CPU)
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Geraet: {DEVICE}")

---
## 1. Daten laden & erkunden

Wir verwenden den **Oxford-IIIT Pet Dataset** (37 Rassen, vereinfacht auf Katze/Hund).

In [ ]:
# --- Konfiguration ---
IMAGE_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = "./data"

# --- Transforms ---
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Lade Oxford-IIIT Pet Dataset...")

In [ ]:
def get_cat_dog_label(label):
    """Katzen-Rassen = Index 0-11, Hunde = 12+"""
    return 0 if label < 12 else 1

class CatDogDataset(torch.utils.data.Dataset):
    """Wrapper: 37 Rassen -> 2 Klassen (Katze/Hund)"""
    def __init__(self, root, split, transform, download=True):
        self.dataset = datasets.OxfordIIITPet(
            root=root, split=split, target_types="category",
            transform=transform, download=download
        )
        # Also keep a version without normalization for visualization
        self.viz_transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor()
        ])
        self.viz_dataset = datasets.OxfordIIITPet(
            root=root, split=split, target_types="category",
            transform=self.viz_transform, download=False
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, get_cat_dog_label(label)

    def get_viz_image(self, idx):
        """Get un-normalized image for display."""
        image, label = self.viz_dataset[idx]
        return image, get_cat_dog_label(label)

# Daten laden
train_full = CatDogDataset(DATA_DIR, "trainval", train_transform)
test_full  = CatDogDataset(DATA_DIR, "test", test_transform)

# Kleines Subset fuer schnelles Training
rng = np.random.default_rng(42)
train_idx = rng.choice(len(train_full), 1000, replace=False)
test_idx  = rng.choice(len(test_full), 200, replace=False)

train_set = Subset(train_full, train_idx)
test_set  = Subset(test_full, test_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f"Trainingsdaten: {len(train_set)} | Testdaten: {len(test_set)}")

### Beispielbilder anzeigen

In [ ]:
CLASS_NAMES = ["Katze", "Hund"]

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    idx = train_idx[i * 80]
    img, label = train_full.get_viz_image(idx)
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(CLASS_NAMES[label], fontsize=12)
    ax.axis("off")
plt.suptitle("Beispielbilder aus dem Datensatz", fontsize=15, fontweight="bold")
plt.tight_layout()

---
## 2. Modell erstellen -- Transfer Learning

Statt ein Netz von Null zu trainieren, nehmen wir ein **vortrainiertes ResNet-18**
(trainiert auf 1.4 Mio. Bildern von ImageNet) und ersetzen nur die letzte Schicht.

| Ansatz | Trainingsdaten noetig | Trainingszeit | Genauigkeit |
|--------|----------------------|---------------|-------------|
| Von Null trainieren | >10.000 | Stunden-Tage | Mittel |
| **Transfer Learning** | **~1.000** | **Minuten** | **Hoch** |

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Alle vortrainierten Schichten einfrieren
for param in model.parameters():
    param.requires_grad = False

# Nur die letzte Schicht ersetzen (2 Ausgaenge: Katze / Hund)
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 2),
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameter gesamt:      {total_params:>10,}")
print(f"Davon trainierbar:     {trainable:>10,}")
print(f"Eingefroren:           {total_params - trainable:>10,}")
print(f"\nWir trainieren nur {trainable/total_params:.2%} der Parameter!")

---
## 3. Training

Jetzt trainieren wir das Modell! Beobachtet, wie **Loss** sinkt und **Accuracy** steigt.

In [ ]:
# --- Hyperparameter (experimentiert damit!) ---
EPOCHS = 5
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

# Training-History speichern
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

print(f"Starte Training auf {DEVICE} ({EPOCHS} Epochen)...\n")

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoche {epoch}/{EPOCHS} [Train]", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # --- Evaluate ---
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    test_loss = running_loss / total
    test_acc = correct / total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"  Epoche {epoch}/{EPOCHS}  |  "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.1%}  |  "
          f"Test  Loss: {test_loss:.4f}  Acc: {test_acc:.1%}")

print(f"\nFertig! Beste Test-Accuracy: {max(history['test_acc']):.1%}")

### Trainings-Kurven

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history["train_loss"], "o-", label="Train", color="#2196F3")
ax1.plot(epochs_range, history["test_loss"], "s-", label="Test", color="#FF5722")
ax1.set_title("Loss (Fehler)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoche")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], "o-", label="Train", color="#2196F3")
ax2.plot(epochs_range, history["test_acc"], "s-", label="Test", color="#FF5722")
ax2.set_title("Accuracy (Genauigkeit)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoche")
ax2.set_ylim(0.4, 1.0)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Trainings-Verlauf", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

### Vorhersagen auf Testbildern

In [ ]:
model.eval()
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(DEVICE)).argmax(1).cpu()
    probs = F.softmax(model(images.to(DEVICE)), dim=1).cpu()

n = min(10, len(images))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat[:n]):
    img = (images[i] * std + mean).clamp(0, 1).permute(1, 2, 0)
    ax.imshow(img)
    color = "green" if preds[i] == labels[i] else "red"
    conf = probs[i, preds[i]].item()
    ax.set_title(f"{CLASS_NAMES[preds[i]]}\n{conf:.0%} sicher", color=color, fontsize=11)
    ax.axis("off")
plt.suptitle("Vorhersagen (Gruen = richtig, Rot = falsch)", fontsize=14, fontweight="bold")
plt.tight_layout()

---
## 4. Black Box vs. Glass Box

### Was bedeutet das?

| | Black Box | Glass Box |
|---|---|---|
| **Prinzip** | Wir sehen nur Input und Output | Wir koennen ins Modell *hineinschauen* |
| **Verstaendnis** | *Was* es entscheidet | *Warum* es so entscheidet |
| **Beispiel** | Das Modell sagt: Hund | Das Modell schaut auf die Ohren |
| **Vertrauen** | Schwer zu beurteilen | Nachvollziehbar |
| **Modelle** | Tiefe Neuronale Netze, LLMs | Entscheidungsbaeume, lineare Modelle |

### Das Problem
Unser ResNet-18 hat **11 Millionen Parameter**. Wie soll ein Mensch verstehen,
warum es bei einem bestimmten Bild "Katze" sagt?

### Die Loesung: Erklaerbare KI (Explainable AI / XAI)
Methoden wie **Grad-CAM** machen die Black Box *glasartig* -- sie zeigen,
**welche Bildregionen** fuer die Entscheidung am wichtigsten waren.

```
        Black Box                       Glass Box
    +------------------+          +------------------+
    | Bild -> ?????? -> Hund |    | Bild -> Ohren -> Hund  |
    |                  |          |        Nase             |
    | (Kein Einblick)  |          |        Fell             |
    +------------------+          +------------------+
```

**Wichtig fuer den Unterricht:** Auch Schueler*innen sollten verstehen,
dass KI-Entscheidungen hinterfragbar sein muessen!

### Black Box Demonstration
Zuerst: Wie fuehlt sich die Black Box an? Wir geben ein Bild rein und bekommen nur eine Antwort.

In [ ]:
# Ein einzelnes Bild durchs Modell schicken -- Black Box Perspektive
sample_img, sample_label = images[0:1].to(DEVICE), labels[0]

with torch.no_grad():
    output = model(sample_img)
    prob = F.softmax(output, dim=1).cpu().squeeze()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1.2]})

# Bild anzeigen
img_show = (images[0] * std + mean).clamp(0, 1).permute(1, 2, 0)
ax1.imshow(img_show)
ax1.set_title("Input", fontsize=13)
ax1.axis("off")

# Wahrscheinlichkeiten als Balkendiagramm
colors = ["#FF6B6B", "#4ECDC4"]
bars = ax2.barh(["Katze", "Hund"], prob.numpy(), color=colors, height=0.5)
ax2.set_xlim(0, 1)
ax2.set_title("Black Box Output", fontsize=13)
ax2.set_xlabel("Wahrscheinlichkeit")
for bar, p in zip(bars, prob):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f"{p:.1%}", va="center", fontsize=12, fontweight="bold")

plt.suptitle('Black Box: Wir sehen nur Input und Output', fontsize=14, fontweight="bold")
plt.tight_layout()

---
## 5. Grad-CAM -- Wo schaut das Modell hin?

**Grad-CAM** (Gradient-weighted Class Activation Mapping) zeigt uns eine **Heatmap**:
Welche Bildbereiche waren fuer die Entscheidung am wichtigsten?

### Wie funktioniert Grad-CAM?
1. Wir berechnen die **Gradienten** des Outputs bezueglich der letzten Faltungsschicht
2. Hohe Gradienten = diese Region war *wichtig* fuer die Entscheidung
3. Wir ueberlagern die Heatmap ueber das Originalbild

Das macht die **Black Box** zur **Glass Box**!

In [ ]:
class GradCAM:
    """Simple Grad-CAM implementation for ResNet."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Hooks registrieren
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        """Generate Grad-CAM heatmap."""
        self.model.eval()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        # Gradienten berechnen
        self.model.zero_grad()
        target = output[0, class_idx]
        target.backward()

        # Gewichtete Kombination der Feature Maps
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # Global Average Pooling
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)  # Nur positive Beitraege
        cam = F.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        # Normieren auf [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())

        return cam, class_idx, F.softmax(output, dim=1).detach().cpu()


# Grad-CAM fuer die letzte Faltungsschicht von ResNet-18 (layer4)
grad_cam = GradCAM(model, model.layer4[-1])
print("Grad-CAM bereit! Zielschicht: model.layer4 (letzte Faltungsschicht)")

### Grad-CAM Heatmaps auf Testbildern

In [ ]:
def show_gradcam_grid(model, grad_cam, images, labels, n=8):
    """Zeigt Originalbilder neben ihren Grad-CAM Heatmaps."""
    n = min(n, len(images))
    fig, axes = plt.subplots(3, n, figsize=(2.8 * n, 8.5))

    for i in range(n):
        img_tensor = images[i:i+1].to(DEVICE)

        # Grad-CAM berechnen
        cam, pred_class, probs = grad_cam.generate(img_tensor)
        conf = probs[0, pred_class].item()

        # Bild de-normalisieren
        img_show = (images[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

        # Row 1: Original
        axes[0, i].imshow(img_show)
        correct = pred_class == labels[i].item()
        color = "green" if correct else "red"
        axes[0, i].set_title(f"{CLASS_NAMES[pred_class]}\n{conf:.0%}", color=color, fontsize=10)
        axes[0, i].axis("off")

        # Row 2: Heatmap
        axes[1, i].imshow(cam, cmap="jet", vmin=0, vmax=1)
        axes[1, i].set_title("Heatmap", fontsize=10)
        axes[1, i].axis("off")

        # Row 3: Overlay
        heatmap_colored = cm.jet(cam)[:, :, :3]
        overlay = 0.5 * img_show + 0.5 * heatmap_colored
        overlay = np.clip(overlay, 0, 1)
        axes[2, i].imshow(overlay)
        axes[2, i].set_title("Overlay", fontsize=10)
        axes[2, i].axis("off")

    axes[0, 0].set_ylabel("Original", fontsize=12, fontweight="bold")
    axes[1, 0].set_ylabel("Grad-CAM", fontsize=12, fontweight="bold")
    axes[2, 0].set_ylabel("Overlay", fontsize=12, fontweight="bold")

    plt.suptitle("Glass Box: Wo schaut das Modell hin?",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()


# Testbilder laden und Grad-CAM anzeigen
test_images, test_labels = next(iter(test_loader))
show_gradcam_grid(model, grad_cam, test_images, test_labels, n=8)

### Einzelbild-Analyse: Was sieht das Modell?

Hier koennen wir ein einzelnes Bild genauer untersuchen. Die Heatmap zeigt,
auf welche Bildbereiche das Modell seinen Blick richtet.

In [ ]:
def detailed_gradcam(model, grad_cam, image, label):
    """Detailansicht eines einzelnen Bildes mit Grad-CAM."""
    img_tensor = image.unsqueeze(0).to(DEVICE)
    cam, pred_class, probs = grad_cam.generate(img_tensor)

    img_show = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    heatmap_colored = cm.jet(cam)[:, :, :3]
    overlay = np.clip(0.5 * img_show + 0.5 * heatmap_colored, 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5),
                              gridspec_kw={"width_ratios": [1, 1, 1, 0.8]})

    axes[0].imshow(img_show)
    axes[0].set_title(f"Original\n(Wahrheit: {CLASS_NAMES[label]})", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(cam, cmap="jet")
    axes[1].set_title("Grad-CAM Heatmap\n(Rot = wichtig)", fontsize=11)
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Overlay", fontsize=11)
    axes[2].axis("off")

    # Wahrscheinlichkeiten
    colors = ["#FF6B6B" if i != pred_class else "#4ECDC4" for i in range(2)]
    bars = axes[3].barh(["Katze", "Hund"], probs[0].numpy(), color=colors, height=0.5)
    axes[3].set_xlim(0, 1)
    axes[3].set_title("Vorhersage", fontsize=11)
    for bar, p in zip(bars, probs[0]):
        axes[3].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                     f"{p:.1%}", va="center", fontsize=11)

    correct_str = "Richtig!" if pred_class == label else "Falsch!"
    plt.suptitle(f"Entscheidung: {CLASS_NAMES[pred_class]} -- {correct_str}",
                 fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout()


# Beispiel: Erstes Testbild detailliert analysieren
detailed_gradcam(model, grad_cam, test_images[0], test_labels[0].item())

In [ ]:
# Noch ein paar weitere Bilder einzeln analysieren
for i in [2, 5, 7]:
    if i < len(test_images):
        detailed_gradcam(model, grad_cam, test_images[i], test_labels[i].item())

---
## 6. Experiment: Was passiert, wenn wir die wichtigen Regionen verdecken?

Wenn Grad-CAM richtig liegt, sollte das Verdecken der wichtigen Regionen
die Vorhersage *verschlechtern*. Das ist ein einfacher **Glass-Box-Test**!

In [ ]:
def occlusion_experiment(model, grad_cam, image, label):
    """Verdeckt die wichtigste Region (laut Grad-CAM) und vergleicht Vorhersagen."""
    img_tensor = image.unsqueeze(0).to(DEVICE)

    # Grad-CAM berechnen
    cam, pred_class, probs_original = grad_cam.generate(img_tensor)

    # Region mit hoechster Aktivierung verdecken (Schwellwert: top 30%)
    mask = cam > np.percentile(cam, 70)
    occluded = image.clone()
    for c in range(3):
        channel = occluded[c]
        channel[torch.from_numpy(mask)] = 0  # Schwarz setzen

    # Neue Vorhersage
    with torch.no_grad():
        probs_occluded = F.softmax(model(occluded.unsqueeze(0).to(DEVICE)), dim=1).cpu()

    img_show = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    occ_show = (occluded * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(img_show)
    axes[0].set_title(f"Original\n{CLASS_NAMES[pred_class]}: {probs_original[0, pred_class]:.1%}",
                      fontsize=11)
    axes[0].axis("off")

    heatmap_colored = cm.jet(cam)[:, :, :3]
    overlay = np.clip(0.5 * img_show + 0.5 * heatmap_colored, 0, 1)
    axes[1].imshow(overlay)
    axes[1].set_title("Wichtige Regionen\n(werden verdeckt)", fontsize=11)
    axes[1].axis("off")

    axes[2].imshow(np.clip(occ_show, 0, 1))
    new_pred = probs_occluded[0].argmax().item()
    axes[2].set_title(f"Verdeckt\n{CLASS_NAMES[new_pred]}: {probs_occluded[0, new_pred]:.1%}",
                      fontsize=11)
    axes[2].axis("off")

    # Confidence change
    delta = probs_original[0, pred_class].item() - probs_occluded[0, pred_class].item()
    plt.suptitle(f"Experiment: Confidence-Aenderung = {delta:+.1%}",
                 fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout()

# Test mit 3 Bildern
for i in [0, 3, 6]:
    if i < len(test_images):
        occlusion_experiment(model, grad_cam, test_images[i], test_labels[i].item())

---
## 7. Zusammenfassung & Diskussion

### Was wir gelernt haben:

| Schritt | Konzept | Analogie im Unterricht |
|---------|---------|------------------------|
| Daten vorbereiten | KI braucht gute Daten | Gute Aufgaben brauchen gutes Material |
| Transfer Learning | Vorwissen nutzen | Vorwissen der Schueler*innen aktivieren |
| Training | Lernen aus Fehlern | Uebung macht den Meister |
| Evaluation | Pruefung auf neuen Daten | Tests zeigen echtes Verstaendnis |
| Grad-CAM | Erklaerbarkeit | Schueler*innen sollen Denkwege zeigen |

### Black Box vs. Glass Box -- Fazit

- **Black Box**: Das Modell sagt Hund -- aber *warum*?
- **Glass Box (Grad-CAM)**: Das Modell schaut auf Ohren, Schnauze, Fell -- *nachvollziehbar*!
- **Occlusion-Test**: Verdecken wir die wichtigen Stellen, wird das Modell unsicher -- die Erklaerung stimmt!

### Diskussionsfragen

1. Schaut das Modell auf die *richtigen* Merkmale? Was waere, wenn es auf den Hintergrund schaut?
2. Wie vertrauenswuerdig ist ein Modell, das wir nicht erklaeren koennen?
3. Wo ist Erklaerbarkeit besonders wichtig? (Medizin, Justiz, Bildung, ...)
4. Sollte KI in der Schule immer *erklaerbar* sein?

In [ ]:
# Modell speichern
torch.save(model.state_dict(), "cat_dog_model.pth")
print("Modell gespeichert als cat_dog_model.pth")
print(f"\nWorkshop beendet! Beste Test-Accuracy: {max(history['test_acc']):.1%}")